## The use of Topological Indices: A Simple QSPR Example

One of the main applications of molecular descriptors is the development of Quantitative Structure-Property Relationships (QSPR), which aim to relate molecular structure to experimentally measurable properties. Rather than performing laboratory measurements for every new compound, molecular descriptors can be used to build predictive models based solely on chemical structure.

In this exercise, we will investigate the relationship between the boiling point of organic compounds and a topological descriptor known as the connectivity index (Chi). Connectivity indices are graph-based descriptors derived from the molecular structure and capture information about molecular size, branching and atomic connectivity without requiring a three-dimensional conformation.

Among these descriptors, Chi1v (the first-order valence connectivity index) is particularly interesting because it reflects the connectivity of pairs of bonded atoms while accounting for their valence states. As molecular size increases, Chi1v generally increases. Conversely, branching tends to modify the connectivity pattern and therefore affects the descriptor value.

<div class="alert alert-block alert-success">
<h2>Contents </h2>

* Retrieve experimental boiling point data from PubChem.
* Calculate the Chi1v descriptor from the molecular structure using RDKit.
* Construct a dataset containing molecular structures, descriptor values and boiling points.
* Visualize the relationship between Chi1v and boiling point using a scatter plot.
* Evaluate whether a simple linear relationship exists between the descriptor and the property.

</div>

### Loading Required Libraries and Defining Helper Functions

Before starting the analysis, we need to import the Python libraries that will be used throughout this practical session.

The imported libraries provide functionality for:

- Data manipulation and analysis (`pandas`)
- Accessing experimental chemical data from PubChem (`requests`)
- Processing molecular structures and calculating descriptors (`RDKit`)
- Data visualization (`matplotlib`)
- Extracting numerical values from text (`re`)

In addition, two helper functions are defined. These functions automate the retrieval of experimental boiling point data from PubChem and the extraction of the numerical boiling point value from the returned information. Defining these functions at the beginning of the notebook allows us to reuse the same code throughout the exercise and keeps the subsequent analysis concise and easier to understand.

Once these tools have been loaded, we will be ready to build our dataset and explore the relationship between molecular descriptors and boiling point.

In [ ]:
import requests
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def get_boiling_point(cid):
    try:
        url = (
            f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/"
            f"compound/{cid}/JSON"
        )
        data = requests.get(url, timeout=30).json()
        sections = data["Record"]["Section"]
        physchem = next(
            s for s in sections
            if s["TOCHeading"] == "Chemical and Physical Properties"
        )
        experimental = next(
            s for s in physchem["Section"]
            if s["TOCHeading"] == "Experimental Properties"
        )
        bp_section = next(
            s for s in experimental["Section"]
            if s["TOCHeading"] == "Boiling Point"
        )
        text = bp_section["Information"][0]["Value"]["StringWithMarkup"][0]["String"]
        match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
        return float(match.group()) if match else None
    except Exception:
        return None
    
def extract_first_number(text):
    if text is None:
        return None
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
    if match:
        return float(match.group())

    return None

### Selection of the Molecular Dataset

Before exploring the relationship between molecular descriptors and boiling point, we must define the set of molecules that will be analyzed.

In this exercise, we will focus on a series of **linear alkanes**, ranging from methane to decane. This family of compounds is particularly suitable for introducing Quantitative Structure-Property Relationships (QSPR) because the molecules are structurally simple and differ mainly in the length of their carbon chain.

As the number of carbon atoms increases, both the molecular size and the strength of intermolecular dispersion forces increase, leading to a gradual increase in boiling point. At the same time, topological descriptors such as the **Chi connectivity indices** also change systematically along the series. This makes linear alkanes an excellent model system for investigating potential relationships between molecular structure and physicochemical properties.

By using a homologous series, we minimize the influence of other structural factors, such as hydrogen bonding, aromaticity, or strong electronic effects, allowing us to focus on the information captured by the molecular descriptors.

#### Retrieving experimental data from PubChem

In this exercise, we will retrieve experimental boiling point data directly from PubChem.

PubChem is one of the largest publicly accessible chemical databases (NCBI). It provides information on millions of chemical compounds, including molecular structures, identifiers, biological activities and experimentally measured physicochemical properties.

To facilitate programmatic access to its contents, PubChem provides a web-based Application Programming Interface (API) known as PUG-REST (Power User Gateway Representational State Transfer). This service allows users to retrieve chemical information directly from Python scripts or other software tools using standard HTTP requests.

In [ ]:
#Get the CIDs for a list of SMILES strings using the PubChem PUG REST API.

pugrest = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
pugin   = "compound/smiles"
pugoper = "cids/txt"

smiles = ["C", "CC", "CCC", "CCCC", "CCCCC", "CCCCCC", "CCCCCCC", "CCCCCCCC", "CCCCCCCCC", "CCCCCCCCCC"]

#For each SMILES string, construct the URL and make the request to get the corresponding CID and save the results in a list.
cids = []
for s in smiles:
    url = "/".join( [pugrest, pugin, s, pugoper] )   # Construct the URL
    res = requests.get(url)
    cids.append(res.text.strip())

df = pd.DataFrame({
    "SMILES": smiles,
    "CID": cids
})

df["BoilingPoint (ºF)"] = df["CID"].apply(get_boiling_point)

print(df)

### Calculating the Molecular Descriptors

A molecular descriptor is a numerical value derived from a molecular structure that captures information about a specific structural or physicochemical characteristic. Descriptors transform chemical structures into numerical variables that can be analysed statistically or used as inputs for predictive models.

#### What is RDKit?
RDKit is one of the most widely used open-source cheminformatics toolkits. It provides a collection of algorithms and data structures for storing, manipulating and analysing chemical structures directly from Python.

RDKit can perform a wide variety of tasks, including:

- Reading and writing molecular structures in different formats.
- Converting SMILES strings into molecular objects.
- Generating molecular fingerprints.
- **Calculating molecular descriptors.**
- Performing substructure searches.
- Visualising chemical structures.
- Preparing data for machine learning applications.

#### The Chi Connectivity Indices
Connectivity indices belong to a family of topological descriptors derived from the molecular graph, where atoms are represented as vertices and bonds as edges.

The first-order valence connectivity index (Chi1v) describes the connectivity of pairs of bonded atoms while taking their valence states into account. Consequently, Chi1v is sensitive to molecular size and branching.

In a homologous series such as the linear alkanes, Chi1v increases regularly as new carbon atoms are added to the chain. Because molecular size is also related to boiling point, a correlation between Chi1v and boiling point may be observed.

Although connectivity indices do not explicitly encode intermolecular interactions, they often act as indirect measures of structural features that influence physical properties.

In [ ]:
# Use RDK to calculate the Chi1v connectivity index for each SMILES string and add it as a new column in the DataFrame.
from rdkit import Chem
from rdkit.Chem.GraphDescriptors import Chi1v

df["Chi1v"] = df["SMILES"].apply(
    lambda smi: Chi1v(Chem.MolFromSmiles(smi))
    if Chem.MolFromSmiles(smi) is not None
    else None
)
print(df)

In [ ]:
# plot the boiling point vs Chi1v

x = df["Chi1v"]
y = df["BoilingPoint (ºF)"]

# Linear regression to fit a line to the data
coef = np.polyfit(x, y, 1)
fit = np.poly1d(coef)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, alpha=0.7)
plt.plot(x, fit(x), color="red")

plt.xlabel("Chi1v")
plt.ylabel("Boiling Point (°F)")
plt.title("Boiling Point vs Chi1v  in alkanes")

plt.grid(True, alpha=0.3)
plt.show()

# print(f"y = {coef[0]:.3f}x + {coef[1]:.3f}")

<div class="alert alert-block alert-warning">
<strong>Further Exploration</strong><br>

Calculate additional molecular descriptors available in RDKit (e.g. Chi0v, Chi1v, MolWt, MolLogP, TPSA and BertzCT).
<br>

1. Determine which descriptor shows the strongest correlation with boiling point.
<br>
2. Calculate the correlation matrix between all descriptors.
<br>
3. Identify pairs of highly correlated descriptors.
<br>
Which descriptor would you select as the best predictor of boiling point? Would you include all descriptors in the same QSPR model?
<br>

<details>
<summary>Solution</summary>
<br>
Many molecular descriptors are correlated because they capture similar aspects of molecular structure. For example, in a homologous series such as the linear alkanes, descriptors related to molecular size (e.g. MolWt and Chi indices) tend to increase simultaneously.
<br>
A descriptor may correlate strongly with a property not because it directly represents the underlying physical phenomenon, but because it is correlated with other structural characteristics that influence that property.
<br>
Including highly correlated descriptors in the same QSPR model may introduce redundancy and reduce model interpretability. Consequently, descriptor selection is an important step in QSPR modelling.

</details>
</div>